# Team 10 데이터 준비 (10/24 토 ~ 10/25 일) — 최지훈 담당 A1·A2

| 순서 | 셀 | 날짜 |
|---|---|---|
| 0 | 설정 (Drive 연결, 경로, 시드) | 공통 |
| 1 | 원본 파일 다운로드 | 10/24 |
| 2 | 원본 검증 (제안서 2.2절 수치와 대조) | 10/24 |
| 3 | 스키마 통일 + ID 매핑 저장 | 10/25 |
| 4 | 상품 메타데이터 → `item_features` | 10/25 |
| 5 | 데이터 설명서(README) 자동 생성 | 10/25 |

- 데이터 출처: Amazon Reviews 2023 (McAuley Lab), 5-core `rating_only` + `last_out_w_his` 분할, `meta_{카테고리}.jsonl.gz`
- **ROOT 경로는 킥오프에서 합의한 공유 폴더로 바꾼 뒤 실행**

In [ ]:
# ===== 0. 설정 =====
import os, gzip, json, re, time, shutil, subprocess
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/Team10_RecSys'   # ← 팀 공유 폴더 경로로 수정
    TMP = '/content/tmp'                            # 다운로드는 Colab 로컬에 먼저 받고 Drive로 복사
except ImportError:                                 # Colab 밖에서 테스트할 때
    ROOT = os.environ.get('TEAM10_ROOT', './Team10_RecSys')
    TMP = os.path.join(ROOT, '_tmp')

RAW  = f'{ROOT}/data/raw'
PROC = f'{ROOT}/data/processed'
DATASETS = ['Video_Games', 'Toys_and_Games']
SEED = 42
COLS = ['user_id', 'parent_asin', 'rating', 'timestamp']   # 원본 컬럼 (history는 읽지 않음)

for ds in DATASETS:
    os.makedirs(f'{RAW}/{ds}', exist_ok=True)
    os.makedirs(f'{PROC}/{ds}', exist_ok=True)
os.makedirs(TMP, exist_ok=True)
print('ROOT =', ROOT)

## 1. 원본 파일 다운로드 (10/24)

데이터셋마다 5개 파일: 전체 5-core 평점 1개, leave-last-out 분할 3개(train/valid/test), 상품 메타데이터 1개.
이미 Drive에 있는 파일은 건너뛰므로 연결이 끊겨도 다시 실행하면 이어서 받는다.

In [ ]:
# ===== 1. 다운로드 =====
BASE = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023'

def file_list(ds):
    return {
        f'{ds}.csv.gz':        f'{BASE}/benchmark/5core/rating_only/{ds}.csv.gz',
        f'{ds}.train.csv.gz':  f'{BASE}/benchmark/5core/last_out_w_his/{ds}.train.csv.gz',
        f'{ds}.valid.csv.gz':  f'{BASE}/benchmark/5core/last_out_w_his/{ds}.valid.csv.gz',
        f'{ds}.test.csv.gz':   f'{BASE}/benchmark/5core/last_out_w_his/{ds}.test.csv.gz',
        f'meta_{ds}.jsonl.gz': f'{BASE}/raw/meta_categories/meta_{ds}.jsonl.gz',
    }

def download_all():
    for ds in DATASETS:
        for name, url in file_list(ds).items():
            dst = f'{RAW}/{ds}/{name}'
            if os.path.exists(dst) and os.path.getsize(dst) > 0:
                print('skip (이미 있음):', name); continue
            tmp = f'{TMP}/{name}'
            print('download:', name)
            subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', tmp, url], check=True)
            shutil.copy(tmp, dst)          # Drive로 복사
            os.remove(tmp)
            print(f'  -> {dst} ({os.path.getsize(dst)/1e6:.1f} MB)')

download_all()

## 2. 원본 검증 (10/24)

제안서 2.2절에서 미리 집계한 수치와 같은지 확인한다. 하나라도 `확인 필요`가 나오면 다음 단계로 넘어가지 말고 원인을 먼저 찾는다.

In [ ]:
# ===== 2. 원본 검증 =====
EXPECTED = {   # 제안서 2.2절 수치
    'Video_Games':    dict(n=814_586,   users=94_762,  items=25_612,  unseen_test_items=345),
    'Toys_and_Games': dict(n=3_861_886, users=432_264, items=162_035, unseen_test_items=1_441),
}

def read_raw(ds):
    dtype = {'user_id': 'string', 'parent_asin': 'string', 'rating': 'float32', 'timestamp': 'int64'}
    allr = pd.read_csv(f'{RAW}/{ds}/{ds}.csv.gz', usecols=COLS, dtype=dtype)
    sp = {s: pd.read_csv(f'{RAW}/{ds}/{ds}.{s}.csv.gz', usecols=COLS, dtype=dtype)
          for s in ['train', 'valid', 'test']}
    return allr, sp

def check(label, ok):
    print(f'  [{"OK" if ok else "확인 필요"}] {label}')
    return ok

def verify(ds):
    print(f'=== {ds} ===')
    allr, sp = read_raw(ds)
    tr, va, te = sp['train'], sp['valid'], sp['test']
    exp = EXPECTED[ds]
    n_users = allr.user_id.nunique()
    results = [
        check(f'전체 평점 수 {len(allr):,} (기대 {exp["n"]:,})', len(allr) == exp['n']),
        check(f'사용자 수 {n_users:,} (기대 {exp["users"]:,})', n_users == exp['users']),
        check(f'상품 수 {allr.parent_asin.nunique():,} (기대 {exp["items"]:,})',
              allr.parent_asin.nunique() == exp['items']),
        check(f'train+valid+test = 전체 ({len(tr)+len(va)+len(te):,})', len(tr)+len(va)+len(te) == len(allr)),
        check('valid·test: 사용자당 정확히 1개',
              va.user_id.is_unique and te.user_id.is_unique and len(va) == len(te) == n_users),
        check(f'train 사용자당 최소 평점 수 = {tr.groupby("user_id").size().min()} (3 이상이어야 함)',
              tr.groupby('user_id').size().min() >= 3),
    ]
    # 시간 순서: train 마지막 <= valid <= test
    t_tr = tr.groupby('user_id').timestamp.max()
    t_va = va.set_index('user_id').timestamp
    t_te = te.set_index('user_id').timestamp
    results.append(check('시간 순서 train ≤ valid ≤ test',
                         bool((t_tr <= t_va.reindex(t_tr.index)).all() and (t_va <= t_te.reindex(t_va.index)).all())))
    # 분할 합집합이 전체 파일과 같은 행인지 (키: user, item, timestamp)
    key = ['user_id', 'parent_asin', 'timestamp']
    merged = pd.concat([tr, va, te])[key].merge(allr[key], on=key, how='outer', indicator=True)
    results.append(check('분할 파일과 전체 파일의 행이 일치', (merged['_merge'] == 'both').all()))
    dup = allr.duplicated(['user_id', 'parent_asin']).sum()
    results.append(check(f'(user, item) 중복 {dup}건', dup == 0))
    unseen = int((~te.parent_asin.isin(set(tr.parent_asin))).sum())   # 제안서 기준: test 행 수
    results.append(check(f'train에 없는 상품이 정답인 test 행 {unseen:,}개 (기대 {exp["unseen_test_items"]:,})',
                         unseen == exp['unseen_test_items']))
    print('  rating 값:', sorted(allr.rating.unique().tolist()))
    print('  timestamp 예시:', allr.timestamp.iloc[0], '→ 13자리면 밀리초(ms)')
    return all(results)

VERIFIED = {ds: verify(ds) for ds in DATASETS}
VERIFIED

## 3. 스키마 통일 + ID 매핑 (10/25)

- 컬럼: `user_id, item_id, rating, timestamp` (`parent_asin` → `item_id`, `history` 제거)
- `timestamp`는 원본 단위(밀리초) 그대로 둔다. 연도가 필요하면 `pd.to_datetime(ts, unit='ms')`
- `item_map`의 `iidx` 순서 = 나중에 만들 TF-IDF 행렬의 행 순서. **팀 전체가 이 번호를 같이 쓴다**

In [ ]:
# ===== 3. 스키마 통일 =====
def unify(df):
    out = df.rename(columns={'parent_asin': 'item_id'})[['user_id', 'item_id', 'rating', 'timestamp']]
    return out.reset_index(drop=True)

def build_processed(ds):
    allr, sp = read_raw(ds)
    out = f'{PROC}/{ds}'
    unify(allr).to_parquet(f'{out}/ratings_all.parquet', index=False)
    for s, df in sp.items():
        unify(df).to_parquet(f'{out}/{s}.parquet', index=False)

    # ID 매핑: 정렬된 고유 ID에 0부터 번호 부여 (누가 돌려도 같은 번호)
    users = np.sort(allr.user_id.unique().astype(str))
    items = np.sort(allr.parent_asin.unique().astype(str))
    pd.DataFrame({'user_id': users, 'uidx': np.arange(len(users), dtype='int32')}) \
      .to_parquet(f'{out}/user_map.parquet', index=False)
    pd.DataFrame({'item_id': items, 'iidx': np.arange(len(items), dtype='int32')}) \
      .to_parquet(f'{out}/item_map.parquet', index=False)

    # 저장한 파일을 다시 읽어서 확인
    tr = pd.read_parquet(f'{out}/train.parquet')
    print(f'{ds}: train {len(tr):,}행, 컬럼 {list(tr.columns)}, dtypes {dict(tr.dtypes.astype(str))}')
    print(f'  user_map {len(users):,}명 / item_map {len(items):,}개 저장 → {out}')

for ds in DATASETS:
    if not VERIFIED.get(ds, False):
        print(f'⚠ {ds}: 2번 검증을 통과하지 못했음. 원인 확인 후 진행'); continue
    build_processed(ds)

## 4. 상품 메타데이터 → `item_features` (10/25)

- 메타 파일 전체를 메모리에 올리지 않고 **한 줄씩** 읽는다 (Toys는 압축을 풀면 2.64GB)
- 줄마다 `parent_asin`만 정규식으로 먼저 꺼내서 5-core 상품이 아니면 `json.loads` 없이 바로 건너뛴다 → 속도 차이가 크다
- 남기는 필드: `title`, `categories`, `features`, `description` (리스트는 공백으로 이어 붙이고 HTML 태그 제거)
- `item_features` = 네 필드를 이어 붙인 텍스트 (TF-IDF 입력)

In [ ]:
# ===== 4. 메타데이터 정리 =====
PID_RE  = re.compile(r'"parent_asin"\s*:\s*"([^"]+)"')
HTML_RE = re.compile(r'<[^>]+>')
WS_RE   = re.compile(r'\s+')
FIELDS  = ['title', 'categories', 'features', 'description']

def to_text(x):
    if x is None:
        return ''
    if isinstance(x, (list, tuple)):
        return ' '.join(to_text(v) for v in x)
    return str(x)

def clean(s):
    return WS_RE.sub(' ', HTML_RE.sub(' ', s)).strip()

def open_meta(ds):
    gz, plain = f'{RAW}/{ds}/meta_{ds}.jsonl.gz', f'{RAW}/{ds}/meta_{ds}.jsonl'
    if os.path.exists(gz):
        return gzip.open(gz, 'rt', encoding='utf-8')
    return open(plain, 'r', encoding='utf-8')   # 압축 푼 파일을 받은 경우

def build_item_features(ds, report_every=200_000):
    items = set(pd.read_parquet(f'{PROC}/{ds}/item_map.parquet')['item_id'])
    rows, seen = {}, 0
    t0 = time.time()
    with open_meta(ds) as f:
        for i, line in enumerate(f, 1):
            m = PID_RE.search(line)
            if m is not None and m.group(1) not in items:
                continue                                   # 5-core 상품 아님 → 파싱 생략
            rec = json.loads(line)
            pid = rec.get('parent_asin')
            if pid not in items or pid in rows:
                continue                                   # 대상 아님 / 중복은 첫 줄만 사용
            rows[pid] = [clean(to_text(rec.get(k))) for k in FIELDS]
            if i % report_every == 0:
                print(f'  {i:,}줄 읽음, 찾은 상품 {len(rows):,}/{len(items):,} ({time.time()-t0:.0f}s)')
    df = pd.DataFrame.from_dict(rows, orient='index', columns=FIELDS)
    df.index.name = 'item_id'
    df = df.reset_index()
    df['item_features'] = (df[FIELDS].agg(' '.join, axis=1)).map(lambda s: WS_RE.sub(' ', s).strip())

    # item_map 순서(iidx)로 정렬해서 저장 → TF-IDF 행 순서와 일치
    imap = pd.read_parquet(f'{PROC}/{ds}/item_map.parquet')
    df = imap.merge(df, on='item_id', how='left')
    for c in FIELDS + ['item_features']:
        df[c] = df[c].fillna('')
    df.to_parquet(f'{PROC}/{ds}/item_features.parquet', index=False)

    found = (df.item_features != '').sum()
    print(f'=== {ds} === {time.time()-t0:.0f}초')
    check(f'메타데이터 확보 {len(rows):,}/{len(items):,}', len(rows) == len(items))
    check(f'item_features 비어 있지 않은 상품 {found:,}/{len(df):,}', found == len(df))
    print('  필드별 빈 비율:', {c: round(float((df[c] == '').mean()), 3) for c in FIELDS})
    print('  텍스트 길이(단어 수) 중앙값:', int(df.item_features.str.split().str.len().median()))
    return df

feat = {ds: build_item_features(ds) for ds in DATASETS}
feat['Video_Games'][['item_id', 'title', 'item_features']].head()

## 5. 데이터 설명서 생성 (10/25)

팀원이 파일을 열기 전에 읽을 `data/processed/README.md`를 자동으로 만든다.

In [ ]:
# ===== 5. README 생성 =====
def write_readme():
    lines = ['# Team 10 processed data', '',
             '출처: Amazon Reviews 2023 (McAuley Lab) 5-core, leave-last-out 공식 분할', '',
             '| 파일 | 컬럼 | 설명 |', '|---|---|---|',
             '| ratings_all / train / valid / test .parquet | user_id, item_id, rating, timestamp | timestamp는 밀리초(ms) Unix time |',
             '| user_map.parquet | user_id, uidx | 정렬된 user_id에 0부터 번호 |',
             '| item_map.parquet | item_id, iidx | 정렬된 item_id에 0부터 번호, TF-IDF 행 순서와 같음 |',
             '| item_features.parquet | item_id, iidx, title, categories, features, description, item_features | iidx 순서로 정렬 |',
             '', '| 데이터셋 | 전체 | train | valid | test | 사용자 | 상품 |', '|---|---|---|---|---|---|---|']
    for ds in DATASETS:
        p = f'{PROC}/{ds}'
        n = {s: len(pd.read_parquet(f'{p}/{s}.parquet', columns=['user_id'])) for s in ['ratings_all','train','valid','test']}
        nu = len(pd.read_parquet(f'{p}/user_map.parquet')); ni = len(pd.read_parquet(f'{p}/item_map.parquet'))
        lines.append(f'| {ds} | {n["ratings_all"]:,} | {n["train"]:,} | {n["valid"]:,} | {n["test"]:,} | {nu:,} | {ni:,} |')
    lines += ['', f'생성: {time.strftime("%Y-%m-%d %H:%M")}, SEED={SEED}']
    with open(f'{PROC}/README.md', 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')
    print('\n'.join(lines))

write_readme()